In [1]:
# ============================================================
# CELL 1: Imports & State Definition
# ============================================================

from typing import TypedDict, Optional, List, Dict, Any, Literal
from enum import Enum
import json
import time
import uuid
from datetime import datetime
from dataclasses import dataclass, field, asdict

# --- STATE DEFINITION (Pydantic-style, strongly typed) ---

class PublishStatus(str, Enum):
    PENDING = "pending"
    PUBLISHING = "publishing"
    PUBLISHED = "published"
    FAILED = "failed"

class WorkflowStatus(str, Enum):
    RUNNING = "running"
    PAUSED = "paused"
    COMPLETED = "completed"
    ERROR = "error"

@dataclass
class WorkflowState:
    session_id: str = ""
    workspace_id: str = ""
    user_id: str = ""
    request: str = ""
    template: Optional[str] = None
    clarification: List[Dict[str, str]] = field(default_factory=list)
    context: Dict[str, Any] = field(default_factory=dict)
    retrieved_docs: List[str] = field(default_factory=list)
    outline: Optional[str] = None
    draft: Optional[str] = None
    images: List[Dict[str, str]] = field(default_factory=list)
    edits: Optional[str] = None
    approved: bool = False
    publish_status: PublishStatus = PublishStatus.PENDING
    usage: Dict[str, Any] = field(default_factory=lambda: {"tokens": 0, "cost": 0.0})
    error: Optional[str] = None
    current_node: str = "START"
    status: WorkflowStatus = WorkflowStatus.RUNNING
    pause_reason: Optional[str] = None
    checkpoint_id: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def to_dict(self):
        d = asdict(self)
        d['publish_status'] = self.publish_status.value
        d['status'] = self.status.value
        return d

    def pretty_print(self):
        print("═" * 60)
        print(f"📋 SESSION: {self.session_id}")
        print(f"📍 NODE: {self.current_node} | STATUS: {self.status.value}")
        if self.pause_reason:
            print(f"⏸️  PAUSE REASON: {self.pause_reason}")
        if self.template:
            print(f"📝 TEMPLATE: {self.template}")
        if self.draft:
            preview = self.draft[:150].replace(chr(10), " ")
            print(f"✏️  DRAFT: {preview}...")
        if self.outline:
            preview = self.outline[:80].replace(chr(10), " ")
            print(f"📑 OUTLINE: {preview}...")
        if self.images:
            print(f"🖼️  IMAGES: {len(self.images)} image(s)")
            for img in self.images:
                print(f"      → {img['url']}")
        if self.error:
            print(f"❌ ERROR: {self.error}")
        print(f"💰 USAGE: {self.usage}")
        print("═" * 60)

print("✅ CELL 1: State definition loaded.")
print("   Run next cell (Mock Workers).")



✅ CELL 1: State definition loaded.
   Run next cell (Mock Workers).


In [2]:
# ============================================================
# CELL 2: Mock Workers (No real API calls)
# ============================================================

class MockWorkers:
    """Mock execution units. Graph calls these, workers don't call each other."""
    
    @staticmethod
    def llm_generate(prompt: str, max_tokens: int = 500) -> str:
        """Mock LLM call — returns fake content"""
        time.sleep(0.2)  # Simulate latency
        
        p = prompt.lower()
        if "caption" in p or "social" in p:
            return """✨ New product launch! Discover the future of AI marketing. 

Transform your content strategy with our AI-powered platform.
🚀 10x faster creation
🎯 Personalized for your audience
📊 Real-time analytics

#AIMarketing #Innovation #FutureOfWork #ContentStrategy"""
        elif "outline" in p:
            return """1. Introduction to AI Marketing
2. Key Benefits for Businesses
   2.1 Automation & Efficiency
   2.2 Personalization at Scale
   2.3 Data-Driven Insights
3. Case Studies & Real Results
4. Implementation Guide
5. Future Trends & Conclusion"""
        elif "blog" in p or "write" in p and "polish" not in p:
            return """# The Future of AI Marketing in 2026

AI marketing is transforming how businesses connect with customers at scale.

## Key Benefits

### Automation
Marketing automation powered by AI reduces manual tasks by 80% while maintaining quality.

### Personalization
AI enables true 1:1 personalization, delivering the right message to the right person at the right time.

## Case Studies

Company X increased engagement by 300% using AI-driven content generation.
Company Y reduced content production costs by 60%.

## Conclusion

The future of marketing is AI-powered, human-supervised."""
        elif "polish" in p:
            return """[POLISHED] The Future of AI Marketing in 2026 | SEO Optimized

AI marketing is revolutionizing customer engagement strategies for modern businesses.

## Key Benefits

### Automation & Efficiency
AI-powered marketing automation reduces manual workload by 80% while maintaining exceptional quality standards.

### Personalization at Scale
Machine learning enables true one-to-one personalization, delivering targeted messages to specific audience segments.

## Real-World Results

- Company X: 300% engagement increase
- Company Y: 60% cost reduction
- Company Z: 5x faster time-to-market

## Implementation Roadmap

1. Audit current workflows
2. Identify automation opportunities
3. Select AI tools
4. Train team
5. Measure & optimize

## Conclusion

AI marketing represents the next evolution of customer engagement."""
        elif "prompt" in p or "image" in p and "generate" not in p:
            return "A futuristic marketing dashboard with holographic charts, neon blue and purple colors, clean minimal design, 4K quality, professional lighting"
        elif "research" in p or "report" in p:
            return """# Research Report: AI Marketing Trends 2026

## Executive Summary
AI marketing adoption has reached 78% among enterprises globally, with average ROI increases of 340%.

## Key Findings
1. 78% of enterprises use AI for content generation
2. Average ROI increase: 340% within 12 months
3. Top use case: Social media content (45%)
4. Second use case: Email personalization (32%)
5. Emerging trend: Video generation (growing 200% YoY)

## Sources
- Gartner Report 2026
- McKinsey Digital Survey
- Internal Enterprise Survey (n=1,000)
- Forrester Wave Analysis

## Confidence Level: HIGH
Data collected from 1,000+ enterprises across 15 countries."""
        elif "clarify" in p:
            return "I need more information: What platform? (Instagram/Twitter/LinkedIn) What tone? (Professional/Casual/Friendly) Who is your target audience?"
        else:
            return f"[MOCK LLM OUTPUT] Generated content for: {prompt[:50]}..."
    
    @staticmethod
    def image_generate(prompt: str) -> Dict[str, str]:
        """Mock image generation"""
        time.sleep(0.3)
        return {
            "url": f"https://mock-cdn.example.com/images/{uuid.uuid4().hex[:8]}.png",
            "prompt": prompt,
            "size": "1024x1024"
        }
    
    @staticmethod
    def web_search(query: str) -> List[str]:
        """Mock web search"""
        time.sleep(0.2)
        return [
            f"[Web] AI Marketing Trends 2026 - {query}",
            f"[Web] Enterprise AI Adoption Report - {query}",
            f"[Web] ROI Analysis of AI Content Tools - {query}",
            f"[Web] Case Study: AI Marketing Success - {query}"
        ]
    
    @staticmethod
    def rag_retrieve(query: str) -> List[str]:
        """Mock RAG retrieval"""
        time.sleep(0.15)
        return [
            "[RAG] Brand Voice Guide: Professional yet approachable tone",
            "[RAG] Product Info: AI Marketing Platform v2.0 features overview",
            "[RAG] Previous Campaign: Summer 2025 results and learnings"
        ]
    
    @staticmethod
    def publish_content(content: str, platform: str) -> Dict[str, Any]:
        """Mock publish"""
        time.sleep(0.2)
        return {
            "published": True,
            "platform": platform,
            "published_at": datetime.now().isoformat(),
            "url": f"https://{platform}.example.com/post/{uuid.uuid4().hex[:8]}"
        }

print("✅ CELL 2: Mock workers loaded.")
print("   Run next cell (Template Subgraphs).")



✅ CELL 2: Mock workers loaded.
   Run next cell (Template Subgraphs).


In [3]:
# ============================================================
# CELL 3: Template Subgraphs (4 templates)
# ============================================================

class TemplateSubgraphs:
    """4 template subgraphs. Each is deterministic, no magic."""
    
    @staticmethod
    def social(state: WorkflowState) -> WorkflowState:
        """SOCIAL: PREPARE → WRITE → OPTIONAL_IMAGE → ASSEMBLE → DONE"""
        print("\n" + "─" * 50)
        print("🟦 [SOCIAL] Starting subgraph...")
        
        # PREPARE
        state.current_node = "SOCIAL_PREPARE"
        print("   📋 PREPARE: Parsing platform, tone, message...")
        platform = state.request.split()[-1] if len(state.request.split()) > 3 else "Instagram"
        state.context['platform'] = platform
        state.context['tone'] = "professional"
        time.sleep(0.1)
        
        # WRITE
        state.current_node = "SOCIAL_WRITE"
        print("   ✍️  WRITE: Generating caption...")
        prompt = f"Write a social media caption for {platform}: {state.request}"
        state.draft = MockWorkers.llm_generate(prompt)
        state.usage['tokens'] += 150
        state.usage['cost'] += 0.003
        print(f"   ✅ Caption generated ({len(state.draft)} chars)")
        
        # OPTIONAL_IMAGE
        state.current_node = "SOCIAL_OPTIONAL_IMAGE"
        if "image" in state.request.lower() or "ảnh" in state.request.lower():
            print("   🖼️  OPTIONAL_IMAGE: Generating image...")
            img_prompt = MockWorkers.llm_generate(f"Create image prompt for: {state.request}")
            img = MockWorkers.image_generate(img_prompt)
            state.images.append(img)
            state.usage['cost'] += 0.02
            print(f"   ✅ Image: {img['url']}")
        else:
            print("   ⏭️  OPTIONAL_IMAGE: Skipped (no image requested)")
        
        # ASSEMBLE
        state.current_node = "SOCIAL_ASSEMBLE"
        print("   🔧 ASSEMBLE: Combining text + images + metadata...")
        state.draft = f"{state.draft}\n\n[Platform: {state.context.get('platform', 'N/A')}]"
        time.sleep(0.1)
        
        # DONE
        state.current_node = "SOCIAL_DONE"
        print("   ✅ [SOCIAL] Subgraph complete!")
        print("─" * 50)
        return state
    
    @staticmethod
    def blog(state: WorkflowState, resume_data: Optional[Dict] = None) -> WorkflowState:
        """BLOG: PREPARE → RAG → OUTLINE → WAIT_APPROVAL → WRITE → POLISH → DONE"""
        print("\n" + "─" * 50)
        print("🟩 [BLOG] Starting subgraph...")
        
        # PREPARE
        state.current_node = "BLOG_PREPARE"
        print("   📋 PREPARE: Topic, audience, SEO keywords...")
        state.context['seo_keywords'] = ['AI marketing', 'automation', 'ROI']
        time.sleep(0.1)
        
        # RAG
        state.current_node = "BLOG_RAG"
        print("   📚 RAG: Retrieving knowledge base...")
        state.retrieved_docs = MockWorkers.rag_retrieve(state.request)
        print(f"   ✅ Retrieved {len(state.retrieved_docs)} docs")
        
        # OUTLINE
        state.current_node = "BLOG_OUTLINE"
        print("   📝 OUTLINE: Generating structure...")
        prompt = f"Create blog outline for: {state.request}"
        state.outline = MockWorkers.llm_generate(prompt)
        state.usage['tokens'] += 200
        state.usage['cost'] += 0.004
        print(f"   ✅ Outline generated ({len(state.outline)} chars)")
        
        # WAIT_APPROVAL — PAUSE POINT
        state.current_node = "BLOG_WAIT_APPROVAL"
        if resume_data is None or resume_data.get('action') != 'approve_outline':
            state.status = WorkflowStatus.PAUSED
            state.pause_reason = "outline_approval"
            state.checkpoint_id = str(uuid.uuid4())[:8]
            print("   ⏸️  WAIT_APPROVAL: PAUSED — Waiting for user to approve outline")
            print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
            print("   👉 User must call resume with: {'action': 'approve_outline'}")
            print("─" * 50)
            return state  # Return early, workflow paused
        
        # User approved outline
        print("   ✅ OUTLINE APPROVED — Continuing...")
        state.status = WorkflowStatus.RUNNING
        state.pause_reason = None
        
        # WRITE
        state.current_node = "BLOG_WRITE"
        print("   ✍️  WRITE: Generating full draft...")
        prompt = f"Write blog based on outline: {state.outline}"
        state.draft = MockWorkers.llm_generate(prompt)
        state.usage['tokens'] += 800
        state.usage['cost'] += 0.016
        print(f"   ✅ Draft generated ({len(state.draft)} chars)")
        
        # POLISH
        state.current_node = "BLOG_POLISH"
        print("   ✨ POLISH: Grammar, SEO, readability...")
        state.draft = MockWorkers.llm_generate(f"Polish this blog: {state.draft[:200]}")
        state.usage['tokens'] += 300
        state.usage['cost'] += 0.006
        print("   ✅ Polished!")
        
        # DONE
        state.current_node = "BLOG_DONE"
        print("   ✅ [BLOG] Subgraph complete!")
        print("─" * 50)
        return state
    
    @staticmethod
    def image(state: WorkflowState) -> WorkflowState:
        """IMAGE: PREPARE → PROMPT → IMAGE → DONE"""
        print("\n" + "─" * 50)
        print("🟨 [IMAGE] Starting subgraph...")
        
        # PREPARE
        state.current_node = "IMAGE_PREPARE"
        print("   📋 PREPARE: Image description, style, size...")
        state.context['style'] = "modern"
        time.sleep(0.1)
        
        # PROMPT
        state.current_node = "IMAGE_PROMPT"
        print("   🎯 PROMPT: Translating to image generation prompt...")
        img_prompt = MockWorkers.llm_generate(f"Create image prompt for: {state.request}")
        state.usage['tokens'] += 100
        state.usage['cost'] += 0.002
        print(f"   ✅ Prompt: {img_prompt[:80]}...")
        
        # IMAGE
        state.current_node = "IMAGE_GENERATE"
        print("   🖼️  IMAGE: Generating image...")
        img = MockWorkers.image_generate(img_prompt)
        state.images.append(img)
        state.draft = f"Image generated: {img['url']}\nPrompt: {img['prompt']}"
        state.usage['cost'] += 0.02
        print(f"   ✅ Image: {img['url']}")
        
        # DONE
        state.current_node = "IMAGE_DONE"
        print("   ✅ [IMAGE] Subgraph complete!")
        print("─" * 50)
        return state
    
    @staticmethod
    def research(state: WorkflowState) -> WorkflowState:
        """RESEARCH: QUERY → SEARCH → RERANK → REPORT → DONE (skip REVIEW)"""
        print("\n" + "─" * 50)
        print("🟪 [RESEARCH] Starting subgraph...")
        
        # QUERY
        state.current_node = "RESEARCH_QUERY"
        print("   📋 QUERY: Parsing research question, scope...")
        state.context['scope'] = "comprehensive"
        time.sleep(0.1)
        
        # SEARCH
        state.current_node = "RESEARCH_SEARCH"
        print("   🔍 SEARCH: Web search + internal KB + RAG...")
        web_results = MockWorkers.web_search(state.request)
        rag_results = MockWorkers.rag_retrieve(state.request)
        state.retrieved_docs = web_results + rag_results
        print(f"   ✅ Found {len(state.retrieved_docs)} sources")
        
        # RERANK
        state.current_node = "RESEARCH_RERANK"
        print("   📊 RERANK: Scoring and filtering...")
        state.retrieved_docs = state.retrieved_docs[:3]
        time.sleep(0.1)
        print(f"   ✅ Top {len(state.retrieved_docs)} sources selected")
        
        # REPORT
        state.current_node = "RESEARCH_REPORT"
        print("   📄 REPORT: Synthesizing findings...")
        prompt = f"Write research report on: {state.request}"
        state.draft = MockWorkers.llm_generate(prompt)
        state.usage['tokens'] += 600
        state.usage['cost'] += 0.012
        print(f"   ✅ Report generated ({len(state.draft)} chars)")
        
        # DONE — Research skips REVIEW/APPROVE/PUBLISH
        state.current_node = "RESEARCH_DONE"
        state.approved = True
        state.publish_status = PublishStatus.PUBLISHED
        print("   ✅ [RESEARCH] Complete! (Auto-approved, skip REVIEW)")
        print("─" * 50)
        return state

print("✅ CELL 3: Template subgraphs loaded.")
print("   Run next cell (Root Graph).")



✅ CELL 3: Template subgraphs loaded.
   Run next cell (Root Graph).


In [4]:
# ============================================================
# CELL 4: Root Graph (Main Orchestrator)
# ============================================================

class RootGraph:
    """
    ROOT GRAPH: Deterministic workflow engine
    START → SESSION_INIT → LOAD_CONTEXT → CLASSIFY_REQUEST → CLARIFY →
    SELECT_TEMPLATE → EXECUTE_TEMPLATE → REVIEW → APPROVE →
    PUBLISH → SAVE → END
    """
    
    def __init__(self):
        self.templates = {
            "social": TemplateSubgraphs.social,
            "blog": TemplateSubgraphs.blog,
            "image": TemplateSubgraphs.image,
            "research": TemplateSubgraphs.research
        }
        self.sessions: Dict[str, WorkflowState] = {}  # In-memory DB
    
    def start(self, request: str, user_id: str = "user_001", workspace_id: str = "ws_001") -> WorkflowState:
        """Start a new workflow session"""
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 12 + "🚀 ROOT GRAPH: NEW WORKFLOW" + " " * 19 + "║")
        print("╚" + "═" * 58 + "╝")
        print(f"   Request: {request[:60]}...")
        
        # SESSION_INIT
        state = WorkflowState()
        state.session_id = str(uuid.uuid4())[:8]
        state.workspace_id = workspace_id
        state.user_id = user_id
        state.request = request
        state.current_node = "SESSION_INIT"
        self.sessions[state.session_id] = state
        print(f"\n📌 SESSION_INIT: {state.session_id}")
        time.sleep(0.1)
        
        # LOAD_CONTEXT
        state.current_node = "LOAD_CONTEXT"
        print("📂 LOAD_CONTEXT: Loading user profile, brand voice, history...")
        state.context = {
            "brand_voice": "Professional yet approachable",
            "user_tier": "pro",
            "history_summary": "Previous campaigns: Summer 2025, Q4 Launch"
        }
        time.sleep(0.1)
        print("   ✅ Context loaded")
        
        # CLASSIFY_REQUEST
        state.current_node = "CLASSIFY_REQUEST"
        print("🔍 CLASSIFY_REQUEST: Analyzing request type...")
        state.template = self._classify(request)
        print(f"   ✅ Detected: {state.template.upper()}")
        
        # CLARIFY
        state.current_node = "CLARIFY"
        print("❓ CLARIFY: Checking if more info needed...")
        needs_clarification = self._needs_clarification(request, state.template)
        
        if needs_clarification:
            state.status = WorkflowStatus.PAUSED
            state.pause_reason = "needs_clarification"
            state.checkpoint_id = str(uuid.uuid4())[:8]
            clarification_q = MockWorkers.llm_generate(f"Clarify: {request}")
            state.clarification.append({"question": clarification_q, "answer": None})
            print(f"   ⏸️  PAUSED — Need clarification: {clarification_q[:60]}...")
            print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
            print("   👉 Call resume with: {'answer': 'your answer'}")
            self.sessions[state.session_id] = state
            return state
        
        print("   ✅ No clarification needed")
        
        # SELECT_TEMPLATE
        state.current_node = "SELECT_TEMPLATE"
        print(f"📋 SELECT_TEMPLATE: {state.template.upper()}")
        
        # EXECUTE_TEMPLATE
        state.current_node = "EXECUTE_TEMPLATE"
        print(f"⚙️  EXECUTE_TEMPLATE: Running {state.template} subgraph...")
        template_fn = self.templates[state.template]
        state = template_fn(state)
        
        # If template paused (e.g., BLOG WAIT_APPROVAL), return early
        if state.status == WorkflowStatus.PAUSED:
            self.sessions[state.session_id] = state
            print(f"\n⏸️  WORKFLOW PAUSED at {state.current_node}")
            print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
            return state
        
        # REVIEW (skip for RESEARCH)
        if state.template != "research":
            return self._handle_review(state)
        else:
            print("\n   ⏭️  REVIEW: Skipped (Research auto-approved)")
            return self._finish_workflow(state)
    
    def resume(self, session_id: str, resume_data: Dict[str, Any]) -> WorkflowState:
        """Resume a paused workflow"""
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 13 + "🔄 ROOT GRAPH: RESUMING" + " " * 20 + "║")
        print("╚" + "═" * 58 + "╝")
        print(f"   Session: {session_id}")
        print(f"   Resume data: {resume_data}")
        
        if session_id not in self.sessions:
            raise ValueError(f"Session {session_id} not found")
        
        state = self.sessions[session_id]
        state.status = WorkflowStatus.RUNNING
        state.pause_reason = None
        
        # Handle different pause reasons
        if state.current_node == "CLARIFY":
            # User answered clarification
            answer = resume_data.get('answer', "")
            state.clarification[-1]['answer'] = answer
            print(f"\n   ✅ Clarification answered: {answer[:60]}...")
            
            # Re-check if still needs clarification
            if self._needs_clarification(state.request, state.template, state.clarification):
                state.status = WorkflowStatus.PAUSED
                state.pause_reason = "needs_clarification"
                state.checkpoint_id = str(uuid.uuid4())[:8]
                new_q = MockWorkers.llm_generate(f"More clarify: {state.request}")
                state.clarification.append({"question": new_q, "answer": None})
                print(f"   ⏸️  Still need more clarification: {new_q[:60]}...")
                self.sessions[session_id] = state
                return state
            
            # Continue from SELECT_TEMPLATE
            print("   ✅ All clarifications complete — Continuing...")
            state.current_node = "SELECT_TEMPLATE"
            print(f"📋 SELECT_TEMPLATE: {state.template.upper()}")
            
            state.current_node = "EXECUTE_TEMPLATE"
            template_fn = self.templates[state.template]
            state = template_fn(state)
            
            if state.status == WorkflowStatus.PAUSED:
                self.sessions[session_id] = state
                return state
            
            return self._handle_review(state)
        
        elif state.current_node == "BLOG_WAIT_APPROVAL":
            # User approved/rejected outline
            action = resume_data.get('action', "")
            if action == "approve_outline":
                print("\n   ✅ Outline approved — Continuing BLOG subgraph...")
                state = TemplateSubgraphs.blog(state, resume_data={'action': 'approve_outline'})
                return self._handle_review(state)
            else:
                print("\n   ❌ Outline rejected — Regenerating...")
                state.edits = resume_data.get('feedback', "")
                state = TemplateSubgraphs.blog(state, resume_data={'action': 'reject_outline'})
                return state
        
        elif state.pause_reason == "review_approval":
            # User reviewed draft
            action = resume_data.get('action', "")
            if action == "approve":
                print("\n   ✅ Draft approved — Continuing to PUBLISH...")
                return self._finish_workflow(state)
            else:
                print("\n   ❌ Draft rejected — Sending edits back...")
                state.edits = resume_data.get('edits', "")
                state.current_node = "EXECUTE_TEMPLATE"
                print("   🔄 Re-running EXECUTE_TEMPLATE with edits...")
                template_fn = self.templates[state.template]
                state = template_fn(state)
                return self._handle_review(state)
        
        self.sessions[session_id] = state
        return state
    
    def _handle_review(self, state: WorkflowState) -> WorkflowState:
        """Handle REVIEW node — pause for user approval"""
        if state.template == "research":
            return self._finish_workflow(state)
        
        state.current_node = "REVIEW"
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 18 + "🔍 REVIEW MODE" + " " * 24 + "║")
        print("╚" + "═" * 58 + "╝")
        preview = state.draft[:200].replace(chr(10), " ") if state.draft else "N/A"
        print(f"   Draft: {preview}...")
        if state.images:
            print(f"   Images: {len(state.images)}")
        print("\n   [APPROVE] → Continue to PUBLISH")
        print("   [REJECT]  → Send edits back to EXECUTE_TEMPLATE")
        
        state.status = WorkflowStatus.PAUSED
        state.pause_reason = "review_approval"
        state.checkpoint_id = str(uuid.uuid4())[:8]
        print(f"\n   ⏸️  PAUSED — Waiting for review")
        print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
        self.sessions[state.session_id] = state
        return state
    
    def _finish_workflow(self, state: WorkflowState) -> WorkflowState:
        """APPROVE → PUBLISH → SAVE → END"""
        
        # APPROVE
        state.current_node = "APPROVE"
        state.approved = True
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 20 + "✅ APPROVED" + " " * 25 + "║")
        print("╚" + "═" * 58 + "╝")
        
        # PUBLISH (idempotent)
        state.current_node = "PUBLISH"
        if state.publish_status == PublishStatus.PUBLISHED:
            print("   ⏭️  PUBLISH: Already published — skipping (idempotent)")
        else:
            print("   📤 PUBLISH: Publishing content...")
            platform = state.context.get('platform', 'generic')
            result = MockWorkers.publish_content(state.draft or "", platform)
            if result['published']:
                state.publish_status = PublishStatus.PUBLISHED
                print(f"   ✅ Published to {result['platform']}: {result['url']}")
            else:
                state.publish_status = PublishStatus.FAILED
                state.error = "Publish failed"
                print("   ❌ Publish failed")
        
        # SAVE
        state.current_node = "SAVE"
        print("   💾 SAVE: Persisting to database...")
        self.sessions[state.session_id] = state
        print("   ✅ Saved")
        
        # END
        state.current_node = "END"
        state.status = WorkflowStatus.COMPLETED
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 16 + "🎉 WORKFLOW COMPLETED" + " " * 19 + "║")
        print("╚" + "═" * 58 + "╝")
        print(f"   Session: {state.session_id}")
        print(f"   Template: {state.template}")
        print(f"   Total cost: ${state.usage['cost']:.4f}")
        print(f"   Total tokens: {state.usage['tokens']}")
        
        return state
    
    def _classify(self, request: str) -> str:
        """Simple keyword-based classification"""
        req = request.lower()
        if any(w in req for w in ['research', 'điều tra', 'đánh giá', 'phân tích', 'báo cáo', 'report']):
            return "research"
        elif any(w in req for w in ['blog', 'bài viết', 'article', 'post dài']):
            return "blog"
        elif any(w in req for w in ['image', 'ảnh', 'hình', 'picture', 'photo']):
            return "image"
        else:
            return "social"
    
    def _needs_clarification(self, request: str, template: str, clarification: List = None) -> bool:
        """Check if request needs more info"""
        if clarification and len(clarification) > 0:
            return False
        return len(request.split()) < 5
    
    def get_status(self, session_id: str) -> Dict[str, Any]:
        """API: GET /workflow/status/{id}"""
        if session_id not in self.sessions:
            return {"error": "Session not found"}
        state = self.sessions[session_id]
        return {
            "session_id": state.session_id,
            "status": state.status.value,
            "current_node": state.current_node,
            "template": state.template,
            "pause_reason": state.pause_reason,
            "checkpoint_id": state.checkpoint_id,
            "draft_preview": state.draft[:200] if state.draft else None,
            "usage": state.usage,
            "error": state.error
        }
    
    def list_sessions(self) -> List[Dict]:
        """API: GET /workspace or GET /history"""
        return [
            {
                "session_id": s.session_id,
                "template": s.template,
                "status": s.status.value,
                "created_at": s.created_at,
                "draft_preview": s.draft[:100] if s.draft else None
            }
            for s in self.sessions.values()
        ]

print("✅ CELL 4: Root Graph loaded.")
print("   Ready to run demos!")
print("\n" + "=" * 60)
print("NEXT: Run DEMO cells below to see workflow in action:")
print("   DEMO 1: Social Media Post")
print("   DEMO 2: Blog Post (with outline approval)")
print("   DEMO 3: Image Generation")
print("   DEMO 4: Research Report")
print("=" * 60)


✅ CELL 4: Root Graph loaded.
   Ready to run demos!

NEXT: Run DEMO cells below to see workflow in action:
   DEMO 1: Social Media Post
   DEMO 2: Blog Post (with outline approval)
   DEMO 3: Image Generation
   DEMO 4: Research Report


In [5]:
# ============================================================
# CELL 5: DEMO 1 — Social Media Post
# Request đủ dài → skip CLARIFY → chạy thẳng → pause ở REVIEW
# ============================================================

print("=" * 60)
print("DEMO 1: SOCIAL MEDIA POST")
print("=" * 60)

graph = RootGraph()

# Step 1: Start workflow
state = graph.start(
    request="Write a social media post about AI marketing tools for Instagram with professional tone and include an image"
)

print("\n" + "━" * 60)
state.pretty_print()

# Step 2: User approves draft (resume)
print("\n" + "━" * 60)
print("👤 USER ACTION: Approving draft...")
print("━" * 60)

final_state = graph.resume(
    session_id=state.session_id,
    resume_data={"action": "approve"}
)

print("\n" + "━" * 60)
final_state.pretty_print()

print("\n✅ DEMO 1 COMPLETE!")



DEMO 1: SOCIAL MEDIA POST

╔══════════════════════════════════════════════════════════╗
║            🚀 ROOT GRAPH: NEW WORKFLOW                   ║
╚══════════════════════════════════════════════════════════╝
   Request: Write a social media post about AI marketing tools for Insta...

📌 SESSION_INIT: 7bdb1291
📂 LOAD_CONTEXT: Loading user profile, brand voice, history...
   ✅ Context loaded
🔍 CLASSIFY_REQUEST: Analyzing request type...
   ✅ Detected: IMAGE
❓ CLARIFY: Checking if more info needed...
   ✅ No clarification needed
📋 SELECT_TEMPLATE: IMAGE
⚙️  EXECUTE_TEMPLATE: Running image subgraph...

──────────────────────────────────────────────────
🟨 [IMAGE] Starting subgraph...
   📋 PREPARE: Image description, style, size...
   🎯 PROMPT: Translating to image generation prompt...
   ✅ Prompt: ✨ New product launch! Discover the future of AI marketing. 

Transform your cont...
   🖼️  IMAGE: Generating image...
   ✅ Image: https://mock-cdn.example.com/images/19c694bd.png
   ✅ [IMAGE] Subg

In [6]:
# ============================================================
# CELL 6: DEMO 2 — BLOG POST
# Request đủ dài → skip CLARIFY → chạy BLOG → PAUSE ở WAIT_APPROVAL (outline)
# User approve outline → chạy tiếp → PAUSE ở REVIEW (draft)
# User approve draft → PUBLISH → SAVE → END
# ============================================================

print("=" * 60)
print("DEMO 2: BLOG POST (with outline approval)")
print("=" * 60)

graph2 = RootGraph()

# Step 1: Start workflow
state2 = graph2.start(
    request="Write a comprehensive blog post about AI marketing trends in 2026 with SEO optimization"
)

print("\n" + "━" * 60)
state2.pretty_print()

# Step 2: User approves outline (resume)
print("\n" + "━" * 60)
print("👤 USER ACTION: Approving outline...")
print("━" * 60)

state2_after_outline = graph2.resume(
    session_id=state2.session_id,
    resume_data={"action": "approve_outline"}
)

print("\n" + "━" * 60)
state2_after_outline.pretty_print()

# Step 3: User approves draft (resume)
print("\n" + "━" * 60)
print("👤 USER ACTION: Approving draft...")
print("━" * 60)

final_state2 = graph2.resume(
    session_id=state2.session_id,
    resume_data={"action": "approve"}
)

print("\n" + "━" * 60)
final_state2.pretty_print()

print("\n✅ DEMO 2 COMPLETE!")

# ============================================================
# CELL 7: DEMO 3 — IMAGE GENERATION
# Simple flow, no pause points
# ============================================================

print("=" * 60)
print("DEMO 3: IMAGE GENERATION")
print("=" * 60)

graph3 = RootGraph()

state3 = graph3.start(
    request="Generate a futuristic marketing dashboard image with neon colors"
)

print("\n" + "━" * 60)
state3.pretty_print()

# Approve and finish
print("\n" + "━" * 60)
print("👤 USER ACTION: Approving draft...")
print("━" * 60)

final_state3 = graph3.resume(
    session_id=state3.session_id,
    resume_data={"action": "approve"}
)

print("\n" + "━" * 60)
final_state3.pretty_print()

print("\n✅ DEMO 3 COMPLETE!")

# ============================================================
# CELL 8: DEMO 4 — RESEARCH REPORT
# Auto-completes, skips REVIEW/APPROVE/PUBLISH
# ============================================================

print("=" * 60)
print("DEMO 4: RESEARCH REPORT (Auto-complete)")
print("=" * 60)

graph4 = RootGraph()

state4 = graph4.start(
    request="Research report on AI marketing ROI for enterprise companies in 2026"
)

print("\n" + "━" * 60)
state4.pretty_print()

print("\n✅ DEMO 4 COMPLETE! (No user action needed)")

# ============================================================
# CELL 9: DASHBOARD — List all sessions
# ============================================================

print("=" * 60)
print("DASHBOARD: All Sessions")
print("=" * 60)

# Combine all sessions from demos
all_sessions = {}
for g in [graph, graph2, graph3, graph4]:
    all_sessions.update(g.sessions)

print(f"\n📊 Total sessions: {len(all_sessions)}\n")
print("─" * 60)

for sid, s in all_sessions.items():
    status_icon = "✅" if s.status.value == "completed" else "⏸️" if s.status.value == "paused" else "🔄"
    print(f"{status_icon} {sid} | {s.template.upper():8} | {s.status.value:10} | {s.current_node}")
    if s.draft:
        preview = s.draft[:60].replace(chr(10), " ")
        print(f"   └─ {preview}...")
    print()

print("─" * 60)
print(f"💰 Total cost across all sessions: ${sum(s.usage['cost'] for s in all_sessions.values()):.4f}")



DEMO 2: BLOG POST (with outline approval)

╔══════════════════════════════════════════════════════════╗
║            🚀 ROOT GRAPH: NEW WORKFLOW                   ║
╚══════════════════════════════════════════════════════════╝
   Request: Write a comprehensive blog post about AI marketing trends in...

📌 SESSION_INIT: 3e064982
📂 LOAD_CONTEXT: Loading user profile, brand voice, history...
   ✅ Context loaded
🔍 CLASSIFY_REQUEST: Analyzing request type...
   ✅ Detected: BLOG
❓ CLARIFY: Checking if more info needed...
   ✅ No clarification needed
📋 SELECT_TEMPLATE: BLOG
⚙️  EXECUTE_TEMPLATE: Running blog subgraph...

──────────────────────────────────────────────────
🟩 [BLOG] Starting subgraph...
   📋 PREPARE: Topic, audience, SEO keywords...
   📚 RAG: Retrieving knowledge base...
   ✅ Retrieved 3 docs
   📝 OUTLINE: Generating structure...
   ✅ Outline generated (238 chars)
   ⏸️  WAIT_APPROVAL: PAUSED — Waiting for user to approve outline
   📌 CHECKPOINT: 945ee0cf
   👉 User must call resume

In [7]:

"""
🤖 AI Marketing Workspace — Mock Workflow System
================================================
Mục đích: Demo toàn bộ workflow deterministics (Root Graph + 4 Templates)
Không cần: API key, LLM, database, Redis, Docker
Chạy: python workflow_mock.py
"""

from typing import Optional, List, Dict, Any
from enum import Enum
import time
import uuid
from datetime import datetime
from dataclasses import dataclass, field, asdict

# ═══════════════════════════════════════════════════════════════
# SECTION 1: STATE DEFINITION
# ═══════════════════════════════════════════════════════════════

class PublishStatus(str, Enum):
    PENDING = "pending"
    PUBLISHING = "publishing"
    PUBLISHED = "published"
    FAILED = "failed"

class WorkflowStatus(str, Enum):
    RUNNING = "running"
    PAUSED = "paused"
    COMPLETED = "completed"
    ERROR = "error"

@dataclass
class WorkflowState:
    session_id: str = ""
    workspace_id: str = ""
    user_id: str = ""
    request: str = ""
    template: Optional[str] = None
    clarification: List[Dict[str, str]] = field(default_factory=list)
    context: Dict[str, Any] = field(default_factory=dict)
    retrieved_docs: List[str] = field(default_factory=list)
    outline: Optional[str] = None
    draft: Optional[str] = None
    images: List[Dict[str, str]] = field(default_factory=list)
    edits: Optional[str] = None
    approved: bool = False
    publish_status: PublishStatus = PublishStatus.PENDING
    usage: Dict[str, Any] = field(default_factory=lambda: {"tokens": 0, "cost": 0.0})
    error: Optional[str] = None
    current_node: str = "START"
    status: WorkflowStatus = WorkflowStatus.RUNNING
    pause_reason: Optional[str] = None
    checkpoint_id: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def pretty_print(self):
        print("═" * 60)
        print(f"📋 SESSION: {self.session_id}")
        print(f"📍 NODE: {self.current_node} | STATUS: {self.status.value}")
        if self.pause_reason:
            print(f"⏸️  PAUSE REASON: {self.pause_reason}")
        if self.template:
            print(f"📝 TEMPLATE: {self.template}")
        if self.draft:
            preview = self.draft[:150].replace(chr(10), " ")
            print(f"✏️  DRAFT: {preview}...")
        if self.outline:
            preview = self.outline[:80].replace(chr(10), " ")
            print(f"📑 OUTLINE: {preview}...")
        if self.images:
            print(f"🖼️  IMAGES: {len(self.images)} image(s)")
            for img in self.images:
                print(f"      → {img['url']}")
        if self.error:
            print(f"❌ ERROR: {self.error}")
        print(f"💰 USAGE: {self.usage}")
        print("═" * 60)


# ═══════════════════════════════════════════════════════════════
# SECTION 2: MOCK WORKERS
# ═══════════════════════════════════════════════════════════════

class MockWorkers:
    """Mock execution units. Graph calls these, workers don't call each other."""
    
    @staticmethod
    def llm_generate(prompt: str, max_tokens: int = 500) -> str:
        time.sleep(0.15)
        p = prompt.lower()
        if "caption" in p or "social" in p:
            return """✨ New product launch! Discover the future of AI marketing. 
Transform your content strategy with our AI-powered platform.
🚀 10x faster creation | 🎯 Personalized | 📊 Real-time analytics
#AIMarketing #Innovation #FutureOfWork"""
        elif "outline" in p:
            return """1. Introduction to AI Marketing
2. Key Benefits for Businesses
   2.1 Automation & Efficiency
   2.2 Personalization at Scale
   2.3 Data-Driven Insights
3. Case Studies & Real Results
4. Implementation Guide
5. Future Trends & Conclusion"""
        elif "blog" in p or "write" in p and "polish" not in p:
            return """# The Future of AI Marketing in 2026
AI marketing is transforming how businesses connect with customers at scale.
## Key Benefits
### Automation
Marketing automation powered by AI reduces manual tasks by 80%.
### Personalization
AI enables true 1:1 personalization at scale.
## Case Studies
Company X increased engagement by 300% using AI-driven content.
## Conclusion
The future of marketing is AI-powered, human-supervised."""
        elif "polish" in p:
            return """[POLISHED] The Future of AI Marketing in 2026 | SEO Optimized
AI marketing is revolutionizing customer engagement strategies.
## Key Benefits
### Automation & Efficiency
AI-powered marketing automation reduces manual workload by 80%.
### Personalization at Scale
Machine learning enables true one-to-one personalization.
## Real-World Results
- Company X: 300% engagement increase
- Company Y: 60% cost reduction
## Implementation Roadmap
1. Audit workflows 2. Identify opportunities 3. Select tools 4. Train team 5. Optimize
## Conclusion
AI marketing represents the next evolution of customer engagement."""
        elif "prompt" in p or ("image" in p and "generate" not in p):
            return "A futuristic marketing dashboard with holographic charts, neon blue and purple colors, clean minimal design, 4K quality"
        elif "research" in p or "report" in p:
            return """# Research Report: AI Marketing Trends 2026
## Executive Summary
AI marketing adoption has reached 78% among enterprises globally.
## Key Findings
1. 78% of enterprises use AI for content generation
2. Average ROI increase: 340% within 12 months
3. Top use case: Social media content (45%)
4. Second use case: Email personalization (32%)
## Sources
- Gartner Report 2026 | McKinsey Digital | Forrester Wave
## Confidence Level: HIGH"""
        elif "clarify" in p:
            return "I need more information: What platform? (Instagram/Twitter/LinkedIn) What tone? (Professional/Casual) Target audience?"
        else:
            return f"[MOCK LLM OUTPUT] Generated content for: {prompt[:50]}..."
    
    @staticmethod
    def image_generate(prompt: str) -> Dict[str, str]:
        time.sleep(0.2)
        return {
            "url": f"https://mock-cdn.example.com/images/{uuid.uuid4().hex[:8]}.png",
            "prompt": prompt,
            "size": "1024x1024"
        }
    
    @staticmethod
    def web_search(query: str) -> List[str]:
        time.sleep(0.15)
        return [
            f"[Web] AI Marketing Trends 2026 - {query}",
            f"[Web] Enterprise AI Adoption Report - {query}",
            f"[Web] ROI Analysis of AI Content Tools - {query}",
        ]
    
    @staticmethod
    def rag_retrieve(query: str) -> List[str]:
        time.sleep(0.1)
        return [
            "[RAG] Brand Voice Guide: Professional yet approachable",
            "[RAG] Product Info: AI Marketing Platform v2.0 features",
            "[RAG] Previous Campaign: Summer 2025 results"
        ]
    
    @staticmethod
    def publish_content(content: str, platform: str) -> Dict[str, Any]:
        time.sleep(0.15)
        return {
            "published": True,
            "platform": platform,
            "published_at": datetime.now().isoformat(),
            "url": f"https://{platform}.example.com/post/{uuid.uuid4().hex[:8]}"
        }


# ═══════════════════════════════════════════════════════════════
# SECTION 3: TEMPLATE SUBGRAPHS
# ═══════════════════════════════════════════════════════════════

class TemplateSubgraphs:
    @staticmethod
    def social(state: WorkflowState) -> WorkflowState:
        print("\n" + "─" * 50)
        print("🟦 [SOCIAL] Starting subgraph...")
        
        state.current_node = "SOCIAL_PREPARE"
        print("   📋 PREPARE: Parsing platform, tone, message...")
        platform = state.request.split()[-1] if len(state.request.split()) > 3 else "Instagram"
        state.context['platform'] = platform
        state.context['tone'] = "professional"
        time.sleep(0.05)
        
        state.current_node = "SOCIAL_WRITE"
        print("   ✍️  WRITE: Generating caption...")
        state.draft = MockWorkers.llm_generate(f"Write caption for {platform}: {state.request}")
        state.usage['tokens'] += 150
        state.usage['cost'] += 0.003
        print(f"   ✅ Caption: {len(state.draft)} chars")
        
        state.current_node = "SOCIAL_OPTIONAL_IMAGE"
        if "image" in state.request.lower():
            print("   🖼️  OPTIONAL_IMAGE: Generating...")
            img = MockWorkers.image_generate(MockWorkers.llm_generate(f"Image prompt for: {state.request}"))
            state.images.append(img)
            state.usage['cost'] += 0.02
            print(f"   ✅ Image: {img['url']}")
        else:
            print("   ⏭️  OPTIONAL_IMAGE: Skipped")
        
        state.current_node = "SOCIAL_ASSEMBLE"
        print("   🔧 ASSEMBLE: Combining...")
        state.draft = f"{state.draft}\\n\\n[Platform: {state.context.get('platform', 'N/A')}]"
        
        state.current_node = "SOCIAL_DONE"
        print("   ✅ [SOCIAL] Complete!")
        print("─" * 50)
        return state
    
    @staticmethod
    def blog(state: WorkflowState, resume_data: Optional[Dict] = None) -> WorkflowState:
        print("\n" + "─" * 50)
        print("🟩 [BLOG] Starting subgraph...")
        
        state.current_node = "BLOG_PREPARE"
        print("   📋 PREPARE: Topic, audience, SEO...")
        state.context['seo_keywords'] = ['AI marketing', 'automation', 'ROI']
        time.sleep(0.05)
        
        state.current_node = "BLOG_RAG"
        print("   📚 RAG: Retrieving...")
        state.retrieved_docs = MockWorkers.rag_retrieve(state.request)
        print(f"   ✅ {len(state.retrieved_docs)} docs")
        
        state.current_node = "BLOG_OUTLINE"
        print("   📝 OUTLINE: Generating...")
        state.outline = MockWorkers.llm_generate(f"Outline for: {state.request}")
        state.usage['tokens'] += 200
        state.usage['cost'] += 0.004
        print(f"   ✅ Outline: {len(state.outline)} chars")
        
        state.current_node = "BLOG_WAIT_APPROVAL"
        if resume_data is None or resume_data.get('action') != 'approve_outline':
            state.status = WorkflowStatus.PAUSED
            state.pause_reason = "outline_approval"
            state.checkpoint_id = str(uuid.uuid4())[:8]
            print("   ⏸️  PAUSED — Waiting for outline approval")
            print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
            print("─" * 50)
            return state
        
        print("   ✅ OUTLINE APPROVED — Continuing...")
        state.status = WorkflowStatus.RUNNING
        state.pause_reason = None
        
        state.current_node = "BLOG_WRITE"
        print("   ✍️  WRITE: Generating draft...")
        state.draft = MockWorkers.llm_generate(f"Write blog: {state.outline}")
        state.usage['tokens'] += 800
        state.usage['cost'] += 0.016
        print(f"   ✅ Draft: {len(state.draft)} chars")
        
        state.current_node = "BLOG_POLISH"
        print("   ✨ POLISH: Optimizing...")
        state.draft = MockWorkers.llm_generate(f"Polish: {state.draft[:200]}")
        state.usage['tokens'] += 300
        state.usage['cost'] += 0.006
        print("   ✅ Polished!")
        
        state.current_node = "BLOG_DONE"
        print("   ✅ [BLOG] Complete!")
        print("─" * 50)
        return state
    
    @staticmethod
    def image(state: WorkflowState) -> WorkflowState:
        print("\n" + "─" * 50)
        print("🟨 [IMAGE] Starting subgraph...")
        
        state.current_node = "IMAGE_PREPARE"
        print("   📋 PREPARE: Description, style...")
        state.context['style'] = "modern"
        time.sleep(0.05)
        
        state.current_node = "IMAGE_PROMPT"
        print("   🎯 PROMPT: Translating...")
        img_prompt = MockWorkers.llm_generate(f"Image prompt for: {state.request}")
        state.usage['tokens'] += 100
        state.usage['cost'] += 0.002
        print(f"   ✅ Prompt: {img_prompt[:60]}...")
        
        state.current_node = "IMAGE_GENERATE"
        print("   🖼️  IMAGE: Generating...")
        img = MockWorkers.image_generate(img_prompt)
        state.images.append(img)
        state.draft = f"Image: {img['url']}\\nPrompt: {img['prompt']}"
        state.usage['cost'] += 0.02
        print(f"   ✅ Image: {img['url']}")
        
        state.current_node = "IMAGE_DONE"
        print("   ✅ [IMAGE] Complete!")
        print("─" * 50)
        return state
    
    @staticmethod
    def research(state: WorkflowState) -> WorkflowState:
        print("\n" + "─" * 50)
        print("🟪 [RESEARCH] Starting subgraph...")
        
        state.current_node = "RESEARCH_QUERY"
        print("   📋 QUERY: Parsing...")
        state.context['scope'] = "comprehensive"
        time.sleep(0.05)
        
        state.current_node = "RESEARCH_SEARCH"
        print("   🔍 SEARCH: Web + KB + RAG...")
        web = MockWorkers.web_search(state.request)
        rag = MockWorkers.rag_retrieve(state.request)
        state.retrieved_docs = web + rag
        print(f"   ✅ {len(state.retrieved_docs)} sources")
        
        state.current_node = "RESEARCH_RERANK"
        print("   📊 RERANK: Filtering...")
        state.retrieved_docs = state.retrieved_docs[:3]
        time.sleep(0.05)
        print(f"   ✅ Top {len(state.retrieved_docs)} selected")
        
        state.current_node = "RESEARCH_REPORT"
        print("   📄 REPORT: Synthesizing...")
        state.draft = MockWorkers.llm_generate(f"Report: {state.request}")
        state.usage['tokens'] += 600
        state.usage['cost'] += 0.012
        print(f"   ✅ Report: {len(state.draft)} chars")
        
        state.current_node = "RESEARCH_DONE"
        state.approved = True
        state.publish_status = PublishStatus.PUBLISHED
        print("   ✅ [RESEARCH] Complete! (Auto-approved)")
        print("─" * 50)
        return state


# ═══════════════════════════════════════════════════════════════
# SECTION 4: ROOT GRAPH
# ═══════════════════════════════════════════════════════════════

class RootGraph:
    def __init__(self):
        self.templates = {
            "social": TemplateSubgraphs.social,
            "blog": TemplateSubgraphs.blog,
            "image": TemplateSubgraphs.image,
            "research": TemplateSubgraphs.research
        }
        self.sessions: Dict[str, WorkflowState] = {}
    
    def start(self, request: str, user_id: str = "user_001", workspace_id: str = "ws_001") -> WorkflowState:
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 12 + "🚀 ROOT GRAPH: NEW WORKFLOW" + " " * 19 + "║")
        print("╚" + "═" * 58 + "╝")
        print(f"   Request: {request[:55]}...")
        
        state = WorkflowState()
        state.session_id = str(uuid.uuid4())[:8]
        state.workspace_id = workspace_id
        state.user_id = user_id
        state.request = request
        state.current_node = "SESSION_INIT"
        self.sessions[state.session_id] = state
        print(f"\n📌 SESSION_INIT: {state.session_id}")
        time.sleep(0.05)
        
        state.current_node = "LOAD_CONTEXT"
        print("📂 LOAD_CONTEXT: Loading profile, brand voice...")
        state.context = {
            "brand_voice": "Professional yet approachable",
            "user_tier": "pro",
            "history_summary": "Previous: Summer 2025, Q4 Launch"
        }
        time.sleep(0.05)
        print("   ✅ Loaded")
        
        state.current_node = "CLASSIFY_REQUEST"
        print("🔍 CLASSIFY_REQUEST: Analyzing...")
        state.template = self._classify(request)
        print(f"   ✅ Detected: {state.template.upper()}")
        
        state.current_node = "CLARIFY"
        print("❓ CLARIFY: Checking...")
        if self._needs_clarification(request, state.template):
            state.status = WorkflowStatus.PAUSED
            state.pause_reason = "needs_clarification"
            state.checkpoint_id = str(uuid.uuid4())[:8]
            q = MockWorkers.llm_generate(f"Clarify: {request}")
            state.clarification.append({"question": q, "answer": None})
            print(f"   ⏸️  PAUSED — Need clarification: {q[:50]}...")
            print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
            self.sessions[state.session_id] = state
            return state
        print("   ✅ No clarification needed")
        
        state.current_node = "SELECT_TEMPLATE"
        print(f"📋 SELECT_TEMPLATE: {state.template.upper()}")
        
        state.current_node = "EXECUTE_TEMPLATE"
        print(f"⚙️  EXECUTE_TEMPLATE: Running {state.template}...")
        state = self.templates[state.template](state)
        
        if state.status == WorkflowStatus.PAUSED:
            self.sessions[state.session_id] = state
            print(f"\n⏸️  PAUSED at {state.current_node}")
            print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
            return state
        
        if state.template != "research":
            return self._handle_review(state)
        else:
            print("\n   ⏭️  REVIEW: Skipped (Research)")
            return self._finish_workflow(state)
    
    def resume(self, session_id: str, resume_data: Dict[str, Any]) -> WorkflowState:
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 13 + "🔄 ROOT GRAPH: RESUMING" + " " * 20 + "║")
        print("╚" + "═" * 58 + "╝")
        print(f"   Session: {session_id}")
        print(f"   Data: {resume_data}")
        
        state = self.sessions[session_id]
        state.status = WorkflowStatus.RUNNING
        state.pause_reason = None
        
        if state.current_node == "CLARIFY":
            answer = resume_data.get('answer', "")
            state.clarification[-1]['answer'] = answer
            print(f"\n   ✅ Answered: {answer[:50]}...")
            if self._needs_clarification(state.request, state.template, state.clarification):
                state.status = WorkflowStatus.PAUSED
                state.pause_reason = "needs_clarification"
                state.checkpoint_id = str(uuid.uuid4())[:8]
                new_q = MockWorkers.llm_generate(f"More: {state.request}")
                state.clarification.append({"question": new_q, "answer": None})
                print(f"   ⏸️  Still need more: {new_q[:50]}...")
                self.sessions[session_id] = state
                return state
            print("   ✅ All clarified — Continuing...")
            state.current_node = "EXECUTE_TEMPLATE"
            state = self.templates[state.template](state)
            if state.status == WorkflowStatus.PAUSED:
                self.sessions[session_id] = state
                return state
            return self._handle_review(state)
        
        elif state.current_node == "BLOG_WAIT_APPROVAL":
            if resume_data.get('action') == 'approve_outline':
                print("\n   ✅ Outline approved — Continuing...")
                state = TemplateSubgraphs.blog(state, {'action': 'approve_outline'})
                return self._handle_review(state)
            else:
                print("\n   ❌ Outline rejected — Regenerating...")
                state.edits = resume_data.get('feedback', "")
                return TemplateSubgraphs.blog(state, {'action': 'reject_outline'})
        
        elif state.pause_reason == "review_approval":
            if resume_data.get('action') == 'approve':
                print("\n   ✅ Draft approved — Publishing...")
                return self._finish_workflow(state)
            else:
                print("\n   ❌ Draft rejected — Re-running...")
                state.edits = resume_data.get('edits', "")
                state.current_node = "EXECUTE_TEMPLATE"
                state = self.templates[state.template](state)
                return self._handle_review(state)
        
        self.sessions[session_id] = state
        return state
    
    def _handle_review(self, state: WorkflowState) -> WorkflowState:
        state.current_node = "REVIEW"
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 18 + "🔍 REVIEW MODE" + " " * 24 + "║")
        print("╚" + "═" * 58 + "╝")
        preview = state.draft[:180].replace(chr(10), " ") if state.draft else "N/A"
        print(f"   Draft: {preview}...")
        if state.images:
            print(f"   Images: {len(state.images)}")
        print("\n   [APPROVE] → PUBLISH  |  [REJECT] → Edit & regenerate")
        state.status = WorkflowStatus.PAUSED
        state.pause_reason = "review_approval"
        state.checkpoint_id = str(uuid.uuid4())[:8]
        print(f"\n   ⏸️  PAUSED — Waiting for review")
        print(f"   📌 CHECKPOINT: {state.checkpoint_id}")
        self.sessions[state.session_id] = state
        return state
    
    def _finish_workflow(self, state: WorkflowState) -> WorkflowState:
        state.current_node = "APPROVE"
        state.approved = True
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 20 + "✅ APPROVED" + " " * 25 + "║")
        print("╚" + "═" * 58 + "╝")
        
        state.current_node = "PUBLISH"
        if state.publish_status == PublishStatus.PUBLISHED:
            print("   ⏭️  Already published — skipping")
        else:
            print("   📤 Publishing...")
            result = MockWorkers.publish_content(state.draft or "", state.context.get('platform', 'generic'))
            state.publish_status = PublishStatus.PUBLISHED if result['published'] else PublishStatus.FAILED
            print(f"   ✅ Published: {result['url']}")
        
        state.current_node = "SAVE"
        print("   💾 Saving...")
        self.sessions[state.session_id] = state
        print("   ✅ Saved")
        
        state.current_node = "END"
        state.status = WorkflowStatus.COMPLETED
        print("\n" + "╔" + "═" * 58 + "╗")
        print("║" + " " * 16 + "🎉 WORKFLOW COMPLETED" + " " * 19 + "║")
        print("╚" + "═" * 58 + "╝")
        print(f"   Session: {state.session_id}")
        print(f"   Template: {state.template}")
        print(f"   Cost: ${state.usage['cost']:.4f} | Tokens: {state.usage['tokens']}")
        return state
    
    def _classify(self, request: str) -> str:
        req = request.lower()
        if any(w in req for w in ['research', 'điều tra', 'đánh giá', 'phân tích', 'báo cáo', 'report']):
            return "research"
        elif any(w in req for w in ['blog', 'bài viết', 'article']):
            return "blog"
        elif any(w in req for w in ['image', 'ảnh', 'hình', 'picture', 'photo']):
            return "image"
        return "social"
    
    def _needs_clarification(self, request: str, template: str, clarification: List = None) -> bool:
        if clarification and len(clarification) > 0:
            return False
        return len(request.split()) < 5


# ═══════════════════════════════════════════════════════════════
# SECTION 5: DEMOS
# ═══════════════════════════════════════════════════════════════

def demo_social():
    print("\\n" + "=" * 60)
    print("DEMO 1: SOCIAL MEDIA POST")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Write a social media post about AI marketing tools for Instagram with image")
    s.pretty_print()
    print("\\n👤 USER: Approving draft...")
    final = g.resume(s.session_id, {"action": "approve"})
    final.pretty_print()
    return g

def demo_blog():
    print("\\n" + "=" * 60)
    print("DEMO 2: BLOG POST (Outline approval)")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Write a comprehensive blog post about AI marketing trends in 2026")
    s.pretty_print()
    print("\\n👤 USER: Approving outline...")
    s2 = g.resume(s.session_id, {"action": "approve_outline"})
    s2.pretty_print()
    print("\\n👤 USER: Approving draft...")
    final = g.resume(s.session_id, {"action": "approve"})
    final.pretty_print()
    return g

def demo_image():
    print("\\n" + "=" * 60)
    print("DEMO 3: IMAGE GENERATION")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Generate a futuristic marketing dashboard image with neon colors")
    s.pretty_print()
    print("\\n👤 USER: Approving draft...")
    final = g.resume(s.session_id, {"action": "approve"})
    final.pretty_print()
    return g

def demo_research():
    print("\\n" + "=" * 60)
    print("DEMO 4: RESEARCH REPORT (Auto-complete)")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Research report on AI marketing ROI for enterprise companies in 2026")
    s.pretty_print()
    print("\\n✅ No user action needed — auto-completed!")
    return g

def dashboard(all_graphs):
    print("\\n" + "=" * 60)
    print("DASHBOARD: All Sessions")
    print("=" * 60)
    all_s = {}
    for g in all_graphs:
        all_s.update(g.sessions)
    print(f"\\n📊 Total: {len(all_s)} sessions\\n")
    for sid, s in all_s.items():
        icon = "✅" if s.status.value == "completed" else "⏸️"
        print(f"{icon} {sid} | {s.template.upper():8} | {s.status.value:10} | {s.current_node}")
        if s.draft:
            print(f"   └─ {s.draft[:55].replace(chr(10), ' ')}...")
    total_cost = sum(s.usage['cost'] for s in all_s.values())
    print(f"\\n💰 Total cost: ${total_cost:.4f}")


# ═══════════════════════════════════════════════════════════════
# MAIN: RUN ALL DEMOS
# ═══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\\n" + "█" * 60)
    print("█" + " " * 15 + "AI MARKETING WORKSPACE" + " " * 21 + "█")
    print("█" + " " * 12 + "Mock Workflow System v1.0" + " " * 21 + "█")
    print("█" * 60)
    
    g1 = demo_social()
    g2 = demo_blog()
    g3 = demo_image()
    g4 = demo_research()
    
    dashboard([g1, g2, g3, g4])
    
    print("\\n" + "█" * 60)
    print("█" + " " * 18 + "ALL DEMOS COMPLETE!" + " " * 19 + "█")
    print("█" * 60)



\n████████████████████████████████████████████████████████████
█               AI MARKETING WORKSPACE                     █
█            Mock Workflow System v1.0                     █
████████████████████████████████████████████████████████████
\n============================================================
DEMO 1: SOCIAL MEDIA POST

╔══════════════════════════════════════════════════════════╗
║            🚀 ROOT GRAPH: NEW WORKFLOW                   ║
╚══════════════════════════════════════════════════════════╝
   Request: Write a social media post about AI marketing tools for ...

📌 SESSION_INIT: c481e1b8
📂 LOAD_CONTEXT: Loading profile, brand voice...
   ✅ Loaded
🔍 CLASSIFY_REQUEST: Analyzing...
   ✅ Detected: IMAGE
❓ CLARIFY: Checking...
   ✅ No clarification needed
📋 SELECT_TEMPLATE: IMAGE
⚙️  EXECUTE_TEMPLATE: Running image...

──────────────────────────────────────────────────
🟨 [IMAGE] Starting subgraph...
   📋 PREPARE: Description, style...
   🎯 PROMPT: Translating...
   ✅ Pro

In [ ]:


from typing import Optional, List, Dict, Any
from enum import Enum
import time
import uuid
from datetime import datetime
from dataclasses import dataclass, field, asdict

# ============================================================
# SECTION 1: STATE DEFINITION
# ============================================================

class PublishStatus(str, Enum):
    PENDING = "pending"
    PUBLISHING = "publishing"
    PUBLISHED = "published"
    FAILED = "failed"

class WorkflowStatus(str, Enum):
    RUNNING = "running"
    PAUSED = "paused"
    COMPLETED = "completed"
    ERROR = "error"

@dataclass
class WorkflowState:
    session_id: str = ""
    workspace_id: str = ""
    user_id: str = ""
    request: str = ""
    template: Optional[str] = None
    clarification: List[Dict[str, str]] = field(default_factory=list)
    context: Dict[str, Any] = field(default_factory=dict)
    retrieved_docs: List[str] = field(default_factory=list)
    outline: Optional[str] = None
    draft: Optional[str] = None
    images: List[Dict[str, str]] = field(default_factory=list)
    edits: Optional[str] = None
    approved: bool = False
    publish_status: PublishStatus = PublishStatus.PENDING
    usage: Dict[str, Any] = field(default_factory=lambda: {"tokens": 0, "cost": 0.0})
    error: Optional[str] = None
    current_node: str = "START"
    status: WorkflowStatus = WorkflowStatus.RUNNING
    pause_reason: Optional[str] = None
    checkpoint_id: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def pretty_print(self):
        print("=" * 60)
        print(f"SESSION: {self.session_id}")
        print(f"NODE: {self.current_node} | STATUS: {self.status.value}")
        if self.pause_reason:
            print(f"PAUSE REASON: {self.pause_reason}")
        if self.template:
            print(f"TEMPLATE: {self.template}")
        if self.draft:
            preview = self.draft[:150].replace("\n", " ")
            print(f"DRAFT: {preview}...")
        if self.outline:
            preview = self.outline[:80].replace("\n", " ")
            print(f"OUTLINE: {preview}...")
        if self.images:
            print(f"IMAGES: {len(self.images)} image(s)")
            for img in self.images:
                print(f"      -> {img['url']}")
        if self.error:
            print(f"ERROR: {self.error}")
        print(f"USAGE: {self.usage}")
        print("=" * 60)


# ============================================================
# SECTION 2: MOCK WORKERS
# ============================================================

class MockWorkers:
    @staticmethod
    def llm_generate(prompt: str, max_tokens: int = 500) -> str:
        time.sleep(0.1)
        p = prompt.lower()
        if "caption" in p or "social" in p:
            return """New product launch! Discover the future of AI marketing. 
Transform your content strategy with our AI-powered platform.
10x faster creation | Personalized | Real-time analytics
#AIMarketing #Innovation #FutureOfWork"""
        elif "outline" in p:
            return """1. Introduction to AI Marketing
2. Key Benefits for Businesses
   2.1 Automation & Efficiency
   2.2 Personalization at Scale
   2.3 Data-Driven Insights
3. Case Studies & Real Results
4. Implementation Guide
5. Future Trends & Conclusion"""
        elif "blog" in p or ("write" in p and "polish" not in p):
            return """# The Future of AI Marketing in 2026
AI marketing is transforming how businesses connect with customers at scale.
## Key Benefits
### Automation
Marketing automation powered by AI reduces manual tasks by 80%.
### Personalization
AI enables true 1:1 personalization at scale.
## Case Studies
Company X increased engagement by 300% using AI-driven content.
## Conclusion
The future of marketing is AI-powered, human-supervised."""
        elif "polish" in p:
            return """[POLISHED] The Future of AI Marketing in 2026 | SEO Optimized
AI marketing is revolutionizing customer engagement strategies.
## Key Benefits
### Automation & Efficiency
AI-powered marketing automation reduces manual workload by 80%.
### Personalization at Scale
Machine learning enables true one-to-one personalization.
## Real-World Results
- Company X: 300% engagement increase
- Company Y: 60% cost reduction
## Implementation Roadmap
1. Audit workflows 2. Identify opportunities 3. Select tools 4. Train team 5. Optimize
## Conclusion
AI marketing represents the next evolution of customer engagement."""
        elif "prompt" in p or ("image" in p and "generate" not in p):
            return "A futuristic marketing dashboard with holographic charts, neon blue and purple colors, clean minimal design, 4K quality"
        elif "research" in p or "report" in p:
            return """# Research Report: AI Marketing Trends 2026
## Executive Summary
AI marketing adoption has reached 78% among enterprises globally.
## Key Findings
1. 78% of enterprises use AI for content generation
2. Average ROI increase: 340% within 12 months
3. Top use case: Social media content (45%)
4. Second use case: Email personalization (32%)
## Sources
- Gartner Report 2026 | McKinsey Digital | Forrester Wave
## Confidence Level: HIGH"""
        elif "clarify" in p:
            return "I need more information: What platform? (Instagram/Twitter/LinkedIn) What tone? (Professional/Casual) Target audience?"
        else:
            return f"[MOCK LLM OUTPUT] Generated content for: {prompt[:50]}..."
    
    @staticmethod
    def image_generate(prompt: str) -> Dict[str, str]:
        time.sleep(0.15)
        return {
            "url": f"https://mock-cdn.example.com/images/{uuid.uuid4().hex[:8]}.png",
            "prompt": prompt,
            "size": "1024x1024"
        }
    
    @staticmethod
    def web_search(query: str) -> List[str]:
        time.sleep(0.1)
        return [
            f"[Web] AI Marketing Trends 2026 - {query}",
            f"[Web] Enterprise AI Adoption Report - {query}",
            f"[Web] ROI Analysis of AI Content Tools - {query}",
        ]
    
    @staticmethod
    def rag_retrieve(query: str) -> List[str]:
        time.sleep(0.08)
        return [
            "[RAG] Brand Voice Guide: Professional yet approachable",
            "[RAG] Product Info: AI Marketing Platform v2.0 features",
            "[RAG] Previous Campaign: Summer 2025 results"
        ]
    
    @staticmethod
    def publish_content(content: str, platform: str) -> Dict[str, Any]:
        time.sleep(0.1)
        return {
            "published": True,
            "platform": platform,
            "published_at": datetime.now().isoformat(),
            "url": f"https://{platform}.example.com/post/{uuid.uuid4().hex[:8]}"
        }


# ============================================================
# SECTION 3: TEMPLATE SUBGRAPHS
# ============================================================

class TemplateSubgraphs:
    @staticmethod
    def social(state: WorkflowState) -> WorkflowState:
        print("\n" + "-" * 50)
        print("[SOCIAL] Starting subgraph...")
        
        state.current_node = "SOCIAL_PREPARE"
        print("   PREPARE: Parsing platform, tone, message...")
        platform = state.request.split()[-1] if len(state.request.split()) > 3 else "Instagram"
        state.context['platform'] = platform
        state.context['tone'] = "professional"
        time.sleep(0.03)
        
        state.current_node = "SOCIAL_WRITE"
        print("   WRITE: Generating caption...")
        state.draft = MockWorkers.llm_generate(f"Write caption for {platform}: {state.request}")
        state.usage['tokens'] += 150
        state.usage['cost'] += 0.003
        print(f"   OK Caption: {len(state.draft)} chars")
        
        state.current_node = "SOCIAL_OPTIONAL_IMAGE"
        if "image" in state.request.lower():
            print("   OPTIONAL_IMAGE: Generating...")
            img = MockWorkers.image_generate(MockWorkers.llm_generate(f"Image prompt for: {state.request}"))
            state.images.append(img)
            state.usage['cost'] += 0.02
            print(f"   OK Image: {img['url']}")
        else:
            print("   OPTIONAL_IMAGE: Skipped")
        
        state.current_node = "SOCIAL_ASSEMBLE"
        print("   ASSEMBLE: Combining...")
        state.draft = f"{state.draft}\n\n[Platform: {state.context.get('platform', 'N/A')}]"
        
        state.current_node = "SOCIAL_DONE"
        print("   [SOCIAL] Complete!")
        print("-" * 50)
        return state
    
    @staticmethod
    def blog(state: WorkflowState, resume_data: Optional[Dict] = None) -> WorkflowState:
        print("\n" + "-" * 50)
        print("[BLOG] Starting subgraph...")
        
        state.current_node = "BLOG_PREPARE"
        print("   PREPARE: Topic, audience, SEO...")
        state.context['seo_keywords'] = ['AI marketing', 'automation', 'ROI']
        time.sleep(0.03)
        
        state.current_node = "BLOG_RAG"
        print("   RAG: Retrieving...")
        state.retrieved_docs = MockWorkers.rag_retrieve(state.request)
        print(f"   OK {len(state.retrieved_docs)} docs")
        
        state.current_node = "BLOG_OUTLINE"
        print("   OUTLINE: Generating...")
        state.outline = MockWorkers.llm_generate(f"Outline for: {state.request}")
        state.usage['tokens'] += 200
        state.usage['cost'] += 0.004
        print(f"   OK Outline: {len(state.outline)} chars")
        
        state.current_node = "BLOG_WAIT_APPROVAL"
        if resume_data is None or resume_data.get('action') != 'approve_outline':
            state.status = WorkflowStatus.PAUSED
            state.pause_reason = "outline_approval"
            state.checkpoint_id = str(uuid.uuid4())[:8]
            print("   PAUSED -- Waiting for outline approval")
            print(f"   CHECKPOINT: {state.checkpoint_id}")
            print("-" * 50)
            return state
        
        print("   OUTLINE APPROVED -- Continuing...")
        state.status = WorkflowStatus.RUNNING
        state.pause_reason = None
        
        state.current_node = "BLOG_WRITE"
        print("   WRITE: Generating draft...")
        state.draft = MockWorkers.llm_generate(f"Write blog: {state.outline}")
        state.usage['tokens'] += 800
        state.usage['cost'] += 0.016
        print(f"   OK Draft: {len(state.draft)} chars")
        
        state.current_node = "BLOG_POLISH"
        print("   POLISH: Optimizing...")
        state.draft = MockWorkers.llm_generate(f"Polish: {state.draft[:200]}")
        state.usage['tokens'] += 300
        state.usage['cost'] += 0.006
        print("   OK Polished!")
        
        state.current_node = "BLOG_DONE"
        print("   [BLOG] Complete!")
        print("-" * 50)
        return state
    
    @staticmethod
    def image(state: WorkflowState) -> WorkflowState:
        print("\n" + "-" * 50)
        print("[IMAGE] Starting subgraph...")
        
        state.current_node = "IMAGE_PREPARE"
        print("   PREPARE: Description, style...")
        state.context['style'] = "modern"
        time.sleep(0.03)
        
        state.current_node = "IMAGE_PROMPT"
        print("   PROMPT: Translating...")
        img_prompt = MockWorkers.llm_generate(f"Image prompt for: {state.request}")
        state.usage['tokens'] += 100
        state.usage['cost'] += 0.002
        print(f"   OK Prompt: {img_prompt[:60]}...")
        
        state.current_node = "IMAGE_GENERATE"
        print("   IMAGE: Generating...")
        img = MockWorkers.image_generate(img_prompt)
        state.images.append(img)
        state.draft = f"Image: {img['url']}\nPrompt: {img['prompt']}"
        state.usage['cost'] += 0.02
        print(f"   OK Image: {img['url']}")
        
        state.current_node = "IMAGE_DONE"
        print("   [IMAGE] Complete!")
        print("-" * 50)
        return state
    
    @staticmethod
    def research(state: WorkflowState) -> WorkflowState:
        print("\n" + "-" * 50)
        print("[RESEARCH] Starting subgraph...")
        
        state.current_node = "RESEARCH_QUERY"
        print("   QUERY: Parsing...")
        state.context['scope'] = "comprehensive"
        time.sleep(0.03)
        
        state.current_node = "RESEARCH_SEARCH"
        print("   SEARCH: Web + KB + RAG...")
        web = MockWorkers.web_search(state.request)
        rag = MockWorkers.rag_retrieve(state.request)
        state.retrieved_docs = web + rag
        print(f"   OK {len(state.retrieved_docs)} sources")
        
        state.current_node = "RESEARCH_RERANK"
        print("   RERANK: Filtering...")
        state.retrieved_docs = state.retrieved_docs[:3]
        time.sleep(0.03)
        print(f"   OK Top {len(state.retrieved_docs)} selected")
        
        state.current_node = "RESEARCH_REPORT"
        print("   REPORT: Synthesizing...")
        state.draft = MockWorkers.llm_generate(f"Report: {state.request}")
        state.usage['tokens'] += 600
        state.usage['cost'] += 0.012
        print(f"   OK Report: {len(state.draft)} chars")
        
        state.current_node = "RESEARCH_DONE"
        state.approved = True
        state.publish_status = PublishStatus.PUBLISHED
        print("   [RESEARCH] Complete! (Auto-approved)")
        print("-" * 50)
        return state


# ============================================================
# SECTION 4: ROOT GRAPH
# ============================================================

class RootGraph:
    def __init__(self):
        self.templates = {
            "social": TemplateSubgraphs.social,
            "blog": TemplateSubgraphs.blog,
            "image": TemplateSubgraphs.image,
            "research": TemplateSubgraphs.research
        }
        self.sessions: Dict[str, WorkflowState] = {}
    
    def start(self, request: str, user_id: str = "user_001", workspace_id: str = "ws_001") -> WorkflowState:
        print("\n" + "+" + "=" * 58 + "+")
        print("|" + " " * 12 + "ROOT GRAPH: NEW WORKFLOW" + " " * 20 + "|")
        print("+" + "=" * 58 + "+")
        print(f"   Request: {request[:55]}...")
        
        state = WorkflowState()
        state.session_id = str(uuid.uuid4())[:8]
        state.workspace_id = workspace_id
        state.user_id = user_id
        state.request = request
        state.current_node = "SESSION_INIT"
        self.sessions[state.session_id] = state
        print(f"\nSESSION_INIT: {state.session_id}")
        time.sleep(0.03)
        
        state.current_node = "LOAD_CONTEXT"
        print("LOAD_CONTEXT: Loading profile, brand voice...")
        state.context = {
            "brand_voice": "Professional yet approachable",
            "user_tier": "pro",
            "history_summary": "Previous: Summer 2025, Q4 Launch"
        }
        time.sleep(0.03)
        print("   OK Loaded")
        
        state.current_node = "CLASSIFY_REQUEST"
        print("CLASSIFY_REQUEST: Analyzing...")
        state.template = self._classify(request)
        print(f"   OK Detected: {state.template.upper()}")
        
        state.current_node = "CLARIFY"
        print("CLARIFY: Checking...")
        if self._needs_clarification(request, state.template):
            state.status = WorkflowStatus.PAUSED
            state.pause_reason = "needs_clarification"
            state.checkpoint_id = str(uuid.uuid4())[:8]
            q = MockWorkers.llm_generate(f"Clarify: {request}")
            state.clarification.append({"question": q, "answer": None})
            print(f"   PAUSED -- Need clarification: {q[:50]}...")
            print(f"   CHECKPOINT: {state.checkpoint_id}")
            self.sessions[state.session_id] = state
            return state
        print("   OK No clarification needed")
        
        state.current_node = "SELECT_TEMPLATE"
        print(f"SELECT_TEMPLATE: {state.template.upper()}")
        
        state.current_node = "EXECUTE_TEMPLATE"
        print(f"EXECUTE_TEMPLATE: Running {state.template}...")
        state = self.templates[state.template](state)
        
        if state.status == WorkflowStatus.PAUSED:
            self.sessions[state.session_id] = state
            print(f"\nPAUSED at {state.current_node}")
            print(f"   CHECKPOINT: {state.checkpoint_id}")
            return state
        
        if state.template != "research":
            return self._handle_review(state)
        else:
            print("\n   REVIEW: Skipped (Research)")
            return self._finish_workflow(state)
    
    def resume(self, session_id: str, resume_data: Dict[str, Any]) -> WorkflowState:
        print("\n" + "+" + "=" * 58 + "+")
        print("|" + " " * 13 + "ROOT GRAPH: RESUMING" + " " * 21 + "|")
        print("+" + "=" * 58 + "+")
        print(f"   Session: {session_id}")
        print(f"   Data: {resume_data}")
        
        state = self.sessions[session_id]
        state.status = WorkflowStatus.RUNNING
        state.pause_reason = None
        
        if state.current_node == "CLARIFY":
            answer = resume_data.get('answer', "")
            state.clarification[-1]['answer'] = answer
            print(f"\n   OK Answered: {answer[:50]}...")
            if self._needs_clarification(state.request, state.template, state.clarification):
                state.status = WorkflowStatus.PAUSED
                state.pause_reason = "needs_clarification"
                state.checkpoint_id = str(uuid.uuid4())[:8]
                new_q = MockWorkers.llm_generate(f"More: {state.request}")
                state.clarification.append({"question": new_q, "answer": None})
                print(f"   PAUSED -- Still need more: {new_q[:50]}...")
                self.sessions[session_id] = state
                return state
            print("   OK All clarified -- Continuing...")
            state.current_node = "EXECUTE_TEMPLATE"
            state = self.templates[state.template](state)
            if state.status == WorkflowStatus.PAUSED:
                self.sessions[session_id] = state
                return state
            return self._handle_review(state)
        
        elif state.current_node == "BLOG_WAIT_APPROVAL":
            if resume_data.get('action') == 'approve_outline':
                print("\n   OK Outline approved -- Continuing...")
                state = TemplateSubgraphs.blog(state, {'action': 'approve_outline'})
                return self._handle_review(state)
            else:
                print("\n   Outline rejected -- Regenerating...")
                state.edits = resume_data.get('feedback', "")
                return TemplateSubgraphs.blog(state, {'action': 'reject_outline'})
        
        elif state.pause_reason == "review_approval":
            if resume_data.get('action') == 'approve':
                print("\n   OK Draft approved -- Publishing...")
                return self._finish_workflow(state)
            else:
                print("\n   Draft rejected -- Re-running...")
                state.edits = resume_data.get('edits', "")
                state.current_node = "EXECUTE_TEMPLATE"
                state = self.templates[state.template](state)
                return self._handle_review(state)
        
        self.sessions[session_id] = state
        return state
    
    def _handle_review(self, state: WorkflowState) -> WorkflowState:
        state.current_node = "REVIEW"
        print("\n" + "+" + "=" * 58 + "+")
        print("|" + " " * 18 + "REVIEW MODE" + " " * 25 + "|")
        print("+" + "=" * 58 + "+")
        preview = state.draft[:180].replace("\n", " ") if state.draft else "N/A"
        print(f"   Draft: {preview}...")
        if state.images:
            print(f"   Images: {len(state.images)}")
        print("\n   [APPROVE] -> PUBLISH  |  [REJECT] -> Edit & regenerate")
        state.status = WorkflowStatus.PAUSED
        state.pause_reason = "review_approval"
        state.checkpoint_id = str(uuid.uuid4())[:8]
        print(f"\nPAUSED -- Waiting for review")
        print(f"   CHECKPOINT: {state.checkpoint_id}")
        self.sessions[state.session_id] = state
        return state
    
    def _finish_workflow(self, state: WorkflowState) -> WorkflowState:
        state.current_node = "APPROVE"
        state.approved = True
        print("\n" + "+" + "=" * 58 + "+")
        print("|" + " " * 20 + "APPROVED" + " " * 26 + "|")
        print("+" + "=" * 58 + "+")
        
        state.current_node = "PUBLISH"
        if state.publish_status == PublishStatus.PUBLISHED:
            print("   Already published -- skipping")
        else:
            print("   Publishing...")
            result = MockWorkers.publish_content(state.draft or "", state.context.get('platform', 'generic'))
            state.publish_status = PublishStatus.PUBLISHED if result['published'] else PublishStatus.FAILED
            print(f"   OK Published: {result['url']}")
        
        state.current_node = "SAVE"
        print("   Saving...")
        self.sessions[state.session_id] = state
        print("   OK Saved")
        
        state.current_node = "END"
        state.status = WorkflowStatus.COMPLETED
        print("\n" + "+" + "=" * 58 + "+")
        print("|" + " " * 16 + "WORKFLOW COMPLETED" + " " * 20 + "|")
        print("+" + "=" * 58 + "+")
        print(f"   Session: {state.session_id}")
        print(f"   Template: {state.template}")
        print(f"   Cost: ${state.usage['cost']:.4f} | Tokens: {state.usage['tokens']}")
        return state
    
    def _classify(self, request: str) -> str:
        req = request.lower()
        if any(w in req for w in ['research', 'report', 'phân tích', 'báo cáo']):
            return "research"
        elif any(w in req for w in ['blog', 'article', 'bài viết']):
            return "blog"
        elif any(w in req for w in ['image', 'ảnh', 'hình', 'picture', 'photo']):
            return "image"
        return "social"
    
    def _needs_clarification(self, request: str, template: str, clarification: List = None) -> bool:
        if clarification and len(clarification) > 0:
            return False
        return len(request.split()) < 5


# ============================================================
# SECTION 5: DEMOS
# ============================================================

def demo_social():
    print("\n" + "=" * 60)
    print("DEMO 1: SOCIAL MEDIA POST")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Write a social media post about AI marketing tools for Instagram with image")
    s.pretty_print()
    print("\nUSER: Approving draft...")
    final = g.resume(s.session_id, {"action": "approve"})
    final.pretty_print()
    return g

def demo_blog():
    print("\n" + "=" * 60)
    print("DEMO 2: BLOG POST (Outline approval)")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Write a comprehensive blog post about AI marketing trends in 2026")
    s.pretty_print()
    print("\nUSER: Approving outline...")
    s2 = g.resume(s.session_id, {"action": "approve_outline"})
    s2.pretty_print()
    print("\nUSER: Approving draft...")
    final = g.resume(s.session_id, {"action": "approve"})
    final.pretty_print()
    return g

def demo_image():
    print("\n" + "=" * 60)
    print("DEMO 3: IMAGE GENERATION")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Generate a futuristic marketing dashboard image with neon colors")
    s.pretty_print()
    print("\nUSER: Approving draft...")
    final = g.resume(s.session_id, {"action": "approve"})
    final.pretty_print()
    return g

def demo_research():
    print("\n" + "=" * 60)
    print("DEMO 4: RESEARCH REPORT (Auto-complete)")
    print("=" * 60)
    g = RootGraph()
    s = g.start("Research report on AI marketing ROI for enterprise companies in 2026")
    s.pretty_print()
    print("\nNo user action needed -- auto-completed!")
    return g

def dashboard(all_graphs):
    print("\n" + "=" * 60)
    print("DASHBOARD: All Sessions")
    print("=" * 60)
    all_s = {}
    for g in all_graphs:
        all_s.update(g.sessions)
    print(f"\nTotal: {len(all_s)} sessions\n")
    for sid, s in all_s.items():
        icon = "OK" if s.status.value == "completed" else "PAUSED"
        print(f"{icon} {sid} | {s.template.upper():8} | {s.status.value:10} | {s.current_node}")
        if s.draft:
            print(f"   -> {s.draft[:55].replace(chr(10), ' ')}...")
    total_cost = sum(s.usage['cost'] for s in all_s.values())
    print(f"\nTotal cost: ${total_cost:.4f}")


# ============================================================
# MAIN: RUN ALL DEMOS
# ============================================================

if __name__ == "__main__":
    print("\n" + "#" * 60)
    print("#" + " " * 15 + "AI MARKETING WORKSPACE" + " " * 21 + "#")
    print("#" + " " * 12 + "Mock Workflow System v1.0" + " " * 21 + "#")
    print("#" * 60)
    
    g1 = demo_social()
    g2 = demo_blog()
    g3 = demo_image()
    g4 = demo_research()
    
    dashboard([g1, g2, g3, g4])
    
    print("\n" + "#" * 60)
    print("#" + " " * 18 + "ALL DEMOS COMPLETE!" + " " * 19 + "#")
    print("#" * 60)



In [ ]:

# Run the script
import subprocess
result = subprocess.run(['python', '/mnt/agents/output/workflow_mock.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
